# 🏋️ NumPy Fundamentals — Practical Exercises

> **Hands-on Practice**: Memory downcasting, 2D slicing, boolean filtering, Z-score normalization aur `argmax` simulation.


## ❓ Question 1: Memory Optimization & Image Pixel Downcasting
- **Problem:** $1000 \times 1000$ image pixels ko `int64` se `uint8` (0-255) me downcast karo.
- **Concept:** RAM memory 8 bytes se 1 byte ho jaati hai (87.5% memory save).


In [1]:
import numpy as np

# 1000 x 1000 image pixel matrix (Total 1 Million pixels)
pixels_64 = np.random.randint(0, 256, size=(1000, 1000), dtype=np.int64)

mem_64 = pixels_64.nbytes / (1024 * 1024)
print(f"Original int64 RAM: {mem_64:.2f} MB")

# Downcast to uint8 (Standard image format)
pixels_uint8 = pixels_64.astype(np.uint8)
mem_uint8 = pixels_uint8.nbytes / (1024 * 1024)

print(f"Optimized uint8 RAM: {mem_uint8:.2f} MB")
percent_saved = ((mem_64 - mem_uint8) / mem_64) * 100
print(f"Total RAM Saved: {percent_saved:.2f}%")
print("Validation Check - Equal values:", np.array_equal(pixels_64, pixels_uint8))


Original int64 RAM: 7.63 MB
Optimized uint8 RAM: 0.95 MB
Total RAM Saved: 87.50%
Validation Check - Equal values: True


## ❓ Question 2: Matrix Reshaping, Slicing & Copy Validation
- **Problem:** 1 se 24 numbers ko $4 \times 6$ matrix me reshape karke center slice extract karo.
- **Concept:** Slice view direct original matrix modify karta hai; `.copy()` original ko protect karta hai.


In [2]:
# 1 se 24 numbers ka array
arr = np.arange(1, 25)

# Reshape into 4x6 matrix
matrix_4x6 = arr.reshape((4, 6))
print("4x6 Matrix:\n", matrix_4x6)

# Center Sub-Matrix Extract karna
center_slice = matrix_4x6[1:3, 2:5]
print("\nExtracted Center Slice (Rows 1:3, Cols 2:5):\n", center_slice)

# View Test: Directly modifying the slice
center_slice[0, 0] = 999
print("\nAfter modifying center_slice[0, 0] = 999:")
print("Original Matrix (Modified via View!):\n", matrix_4x6)

# Safe Copy Test
matrix_copy = matrix_4x6.copy()
safe_slice = matrix_copy[1:3, 2:5].copy()
safe_slice[0, 0] = 111
print("\nSafe Copy modification test:")
print("Original Matrix remaining unchanged at [1, 2]:", matrix_4x6[1, 2])


4x6 Matrix:
 [[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]
 [13 14 15 16 17 18]
 [19 20 21 22 23 24]]

Extracted Center Slice (Rows 1:3, Cols 2:5):
 [[ 9 10 11]
 [15 16 17]]

After modifying center_slice[0, 0] = 999:
Original Matrix (Modified via View!):
 [[  1   2   3   4   5   6]
 [  7   8 999  10  11  12]
 [ 13  14  15  16  17  18]
 [ 19  20  21  22  23  24]]

Safe Copy modification test:
Original Matrix remaining unchanged at [1, 2]: 999


## ❓ Question 3: Outlier Detection & Conditional Filtering (Boolean Masking)
- **Problem:** 30 temperature readings me se $>38^\circ\text{C}$ alert filter karo aur $>40^\circ\text{C}$ clip karo.
- **Concept:** Fast in-place vectorized clipping bina kisi loop ke `arr[arr > 40] = 40.0`.


In [3]:
np.random.seed(42)

# 30 temperature readings (Float values)
temperatures = np.round(np.random.uniform(15.0, 45.0, size=30), 1)
print("Original Sensor Readings:\n", temperatures)

# 1. Heat Alert Readings (> 38°C)
heat_alerts = temperatures[temperatures > 38.0]
print(f"\nHeat Alerts (> 38°C) Total Count: {len(heat_alerts)}")
print("Alert Values:", heat_alerts)

# 2. Outlier Clipping: Values > 40 ko 40.0 par clip karna
temperatures[temperatures > 40.0] = 40.0
print("\nCleaned & Clipped Readings (Max cap 40.0):\n", temperatures)
print("Max value after clipping:", np.max(temperatures))


Original Sensor Readings:
 [26.2 43.5 37.  33.  19.7 19.7 16.7 41.  33.  36.2 15.6 44.1 40.  21.4
 20.5 20.5 24.1 30.7 28.  23.7 33.4 19.2 23.8 26.  28.7 38.6 21.  30.4
 32.8 16.4]

Heat Alerts (> 38°C) Total Count: 5
Alert Values: [43.5 41.  44.1 40.  38.6]

Cleaned & Clipped Readings (Max cap 40.0):
 [26.2 40.  37.  33.  19.7 19.7 16.7 40.  33.  36.2 15.6 40.  40.  21.4
 20.5 20.5 24.1 30.7 28.  23.7 33.4 19.2 23.8 26.  28.7 38.6 21.  30.4
 32.8 16.4]
Max value after clipping: 40.0


## ❓ Question 4: Feature-wise Z-Score Normalization with Broadcasting
- **Problem:** 4 students ke test scores (scales 50, 100, 500) ko column-wise normalize karo.
- **Concept:** `axis=0` mean aur std calculate karke broadcasting se scale alignment $Z = (X - \mu)/\sigma$.


In [4]:
scores = np.array([
    [35.0, 78.0, 340.0],
    [45.0, 92.0, 480.0],
    [20.0, 55.0, 220.0],
    [40.0, 85.0, 410.0]
])

print("Raw Scores (Rows=Students, Cols=Subjects):\n", scores)

col_means = np.mean(scores, axis=0)
col_stds = np.std(scores, axis=0)

print("\nSubject-wise Means:", col_means)
print("Subject-wise Std Devs:", col_stds)

normalized_scores = (scores - col_means) / col_stds

print("\n--- Normalized Scores Matrix ---")
print(np.round(normalized_scores, 3))
print("\nVerification - Column Means (~0):", np.round(np.mean(normalized_scores, axis=0), 4))
print("Verification - Column Std Devs (~1):", np.round(np.std(normalized_scores, axis=0), 4))


Raw Scores (Rows=Students, Cols=Subjects):
 [[ 35.  78. 340.]
 [ 45.  92. 480.]
 [ 20.  55. 220.]
 [ 40.  85. 410.]]

Subject-wise Means: [ 35.   77.5 362.5]
Subject-wise Std Devs: [ 9.35414347 13.90143877 96.01432185]

--- Normalized Scores Matrix ---
[[ 0.     0.036 -0.234]
 [ 1.069  1.043  1.224]
 [-1.604 -1.619 -1.484]
 [ 0.535  0.54   0.495]]

Verification - Column Means (~0): [ 0.  0. -0.]
Verification - Column Std Devs (~1): [1. 1. 1.]


## ❓ Question 5: Statistical Analysis & Probability ArgMax Simulation
- **Problem:** 4 images ke 5 class probabilities me se highest confidence class index find out karo.
- **Concept:** `np.argmax(probs, axis=1)` row-wise best prediction return karta hai.


In [5]:
np.random.seed(101)
raw_probs = np.random.uniform(0.1, 0.9, size=(4, 5))
probs = raw_probs / np.sum(raw_probs, axis=1, keepdims=True)

class_names = ["Cat", "Dog", "Bird", "Horse", "Fish"]

print("Model Probabilities Matrix (4 Images x 5 Classes):\n", np.round(probs, 3))

# 1. Best Class Prediction using argmax along axis=1
predicted_indices = np.argmax(probs, axis=1)
predicted_classes = [class_names[idx] for idx in predicted_indices]

print("\n--- Predictions ---")
for img_num, (idx, name) in enumerate(zip(predicted_indices, predicted_classes)):
    confidence = probs[img_num, idx]
    print(f"Image {img_num + 1}: Predicted Class = '{name}' (Index: {idx}) with Confidence = {confidence:.2%}")

# 2. Overall Statistical Summary
all_confs = np.max(probs, axis=1)
print("\n--- Model Confidence Statistics ---")
print("Mean Confidence:      ", round(float(np.mean(all_confs)), 3))
print("Median Confidence:    ", round(float(np.median(all_confs)), 3))
print("Min Confidence:       ", round(float(np.min(all_confs)), 3))
print("Max Confidence:       ", round(float(np.max(all_confs)), 3))
print("Confidence Variance:  ", round(float(np.var(all_confs)), 5))


Model Probabilities Matrix (4 Images x 5 Classes):
 [[0.247 0.268 0.059 0.114 0.312]
 [0.269 0.121 0.285 0.237 0.088]
 [0.196 0.138 0.089 0.263 0.315]
 [0.14  0.082 0.286 0.335 0.157]]

--- Predictions ---
Image 1: Predicted Class = 'Fish' (Index: 4) with Confidence = 31.20%
Image 2: Predicted Class = 'Bird' (Index: 2) with Confidence = 28.52%
Image 3: Predicted Class = 'Fish' (Index: 4) with Confidence = 31.48%
Image 4: Predicted Class = 'Horse' (Index: 3) with Confidence = 33.49%

--- Model Confidence Statistics ---
Mean Confidence:       0.312
Median Confidence:     0.313
Min Confidence:        0.285
Max Confidence:        0.335
Confidence Variance:   0.00031
